# MODULE 5 — STATISTICAL LANGUAGE MODELS & N-GRAMS
**Natural Language Processing | Phenikaa University**  
**Demo • Guided Practice • Exercises**

**Nội dung:** Unigram / Bigram / Trigram, chain rule, Markov assumption, MLE, sparsity, Laplace/Add-$k$, back-off, interpolation, perplexity, text generation.

**Kết quả học tập:** Sau buổi học, sinh viên có thể (1) xây dựng N-gram từ corpus; (2) ước lượng xác suất và xử lý zero-count; (3) đánh giá bằng perplexity trên tập test độc lập; (4) sinh văn bản theo phân phối xác suất.

> Chạy lần lượt các cell từ trên xuống. Notebook chạy offline với thư viện Python chuẩn; không cần cài thêm package. Các bài tập ở cuối có phần TODO để sinh viên hoàn thiện.

## 1. Bài toán mô hình ngôn ngữ
Một **language model** gán xác suất cho chuỗi $W=(w_1,\ldots,w_T)$ hoặc dự đoán từ tiếp theo $P(w_t\mid w_1,\ldots,w_{t-1})$.

**Chain rule** (đẳng thức chính xác):
$$P(w_1,\ldots,w_T)=\prod_{t=1}^{T}P(w_t\mid w_1,\ldots,w_{t-1}).$$

**Giả định Markov bậc $n-1$** (xấp xỉ):
$$P(w_t\mid w_1,\ldots,w_{t-1})\approx P(w_t\mid w_{t-n+1},\ldots,w_{t-1}).$$

- Unigram ($n=1$): không dùng lịch sử.
- Bigram ($n=2$): dùng 1 từ trước.
- Trigram ($n=3$): dùng 2 từ trước.

Ta thêm token bắt đầu câu `<s>` và token kết thúc `</s>`. Với trigram cần **hai** `<s>` tại đầu mỗi câu. `<s>` chỉ là lịch sử, không được sinh ra như một từ bình thường.

In [1]:
from collections import Counter, defaultdict
from math import log2, exp2, isfinite
import random

SEED = 42
random.seed(SEED)
# Corpus đơn giản để dễ kiểm tra thủ công; mỗi phần tử là một câu đã tách token.
train_sentences = [
    'tôi thích học máy', 'tôi thích xử lý ngôn ngữ',
    'tôi học xử lý ngôn ngữ', 'bạn thích học máy',
    'bạn học ngôn ngữ tự nhiên', 'tôi thích ngôn ngữ tự nhiên',
    'mô hình ngôn ngữ dự đoán từ', 'mô hình học từ dữ liệu',
    'học máy xử lý dữ liệu', 'ngôn ngữ tự nhiên rất thú vị',
]
# Tách dữ liệu câu trước khi xây dựng từ vựng / các bộ đếm.
dev_sentences = ['tôi học ngôn ngữ tự nhiên', 'bạn thích xử lý ngôn ngữ']
test_sentences = ['tôi thích học ngôn ngữ', 'mô hình xử lý dữ liệu']
print('Số câu train / dev / test:', len(train_sentences), len(dev_sentences), len(test_sentences))
print('Ví dụ:', train_sentences[0])

Số câu train / dev / test: 10 2 2
Ví dụ: tôi thích học máy


### Lưu ý riêng cho tiếng Việt
Ví dụ sử dụng `str.split()` nên token được tính theo **âm tiết cách nhau bằng dấu cách**, chưa phải từ tiếng Việt phân đoạn chuẩn. Trong bài tập mở rộng với *Truyện Kiều*, sinh viên phải xác định rõ mức token: âm tiết hay từ đã tách từ. Không bỏ stopwords khi huấn luyện mô hình ngôn ngữ thông thường vì các từ chức năng ảnh hưởng xác suất chuỗi.

In [2]:
def tokenize(sentence):
    return sentence.lower().strip().split()

def make_vocabulary(sentences, min_count=1):
    counts = Counter(w for s in sentences for w in tokenize(s))
    # </s> là một kết quả dự đoán hợp lệ; <unk> dành cho từ ngoài vocab.
    return {w for w, c in counts.items() if c >= min_count} | {'</s>', '<unk>'}

vocab = make_vocabulary(train_sentences)
predictable = sorted(vocab)   # KHÔNG chứa <s>
print('Vocabulary size =', len(vocab))
print(predictable)

def prepare(sentence, order, vocabulary):
    words = [w if w in vocabulary else '<unk>' for w in tokenize(sentence)]
    return ['<s>'] * (order - 1) + words + ['</s>']

for order in (1, 2, 3):
    print(f'{order}-gram:', prepare(train_sentences[0], order, vocab))

Vocabulary size = 23
['</s>', '<unk>', 'bạn', 'dữ', 'dự', 'hình', 'học', 'liệu', 'lý', 'máy', 'mô', 'ngôn', 'ngữ', 'nhiên', 'rất', 'thích', 'thú', 'tôi', 'từ', 'tự', 'vị', 'xử', 'đoán']
1-gram: ['tôi', 'thích', 'học', 'máy', '</s>']
2-gram: ['<s>', 'tôi', 'thích', 'học', 'máy', '</s>']
3-gram: ['<s>', '<s>', 'tôi', 'thích', 'học', 'máy', '</s>']


## 2. Đếm N-gram và ước lượng MLE
Với lịch sử $h$ (gồm $n-1$ token), $C(h,w)$ là số lần xuất hiện của N-gram $(h,w)$, và $C(h)=\sum_{w\in V} C(h,w)$ là số lần lịch sử $h$ được dùng để dự đoán token tiếp theo.

$$P_{\mathrm{MLE}}(w\mid h)=\frac{C(h,w)}{C(h)},\qquad C(h)>0.$$

Lưu ý: đối với bigram/trigram, mẫu số phải là tổng số **cơ hội dự đoán** sau $h$; không được mặc nhiên dùng số lần $h$ xuất hiện mà không xét token kết thúc câu.

In [3]:
def fit_counts(sentences, order, vocabulary):
    counts = Counter()
    history_counts = Counter()
    for sentence in sentences:
        tokens = prepare(sentence, order, vocabulary)
        for i in range(order - 1, len(tokens)):
            history = tuple(tokens[i - order + 1:i]) if order > 1 else ()
            word = tokens[i]
            counts[history + (word,)] += 1
            history_counts[history] += 1
    return counts, history_counts

models = {order: fit_counts(train_sentences, order, vocab) for order in (1, 2, 3)}

def mle_prob(word, history, order):
    counts, history_counts = models[order]
    h = tuple(history[-(order-1):]) if order > 1 else ()
    denom = history_counts[h]
    return counts[h + (word,)] / denom if denom else 0.0

for order in (1, 2, 3):
    counts, hc = models[order]
    print(f'Order={order}: {len(counts)} n-gram types, {sum(counts.values())} prediction events')
print('P(thích | tôi) =', mle_prob('thích', ('tôi',), 2))
print('P(học | tôi, thích) =', mle_prob('học', ('tôi','thích'), 3))

Order=1: 22 n-gram types, 68 prediction events
Order=2: 40 n-gram types, 68 prediction events
Order=3: 49 n-gram types, 68 prediction events
P(thích | tôi) = 0.75
P(học | tôi, thích) = 0.3333333333333333


### Kiểm chứng: tổng xác suất theo một lịch sử bằng 1
Với mọi $h$ đã thấy trong dữ liệu huấn luyện:
$$\sum_{w\in V}P_{\mathrm{MLE}}(w\mid h)=1.$$
Trường hợp $C(h)=0$, MLE **không xác định** và code trả 0 như một quy ước kỹ thuật, không phải một phân phối hợp lệ.

In [4]:
for order in (1, 2, 3):
    _, hc = models[order]
    for history in hc:
        total = sum(mle_prob(word, history, order) for word in predictable)
        assert abs(total - 1) < 1e-10, (order, history, total)
print('PASS: Tất cả phân phối MLE tại lịch sử đã quan sát đều chuẩn hóa.')

# Tự kiểm chứng xác suất bằng tần suất:
bi_counts, bi_hc = models[2]
print('C(tôi, thích) =', bi_counts[('tôi','thích')])
print('C(tôi) (vị trí dự đoán được) =', bi_hc[('tôi',)])

PASS: Tất cả phân phối MLE tại lịch sử đã quan sát đều chuẩn hóa.
C(tôi, thích) = 3
C(tôi) (vị trí dự đoán được) = 4


## 3. Vấn đề zero-count và data sparsity
N-gram chưa xuất hiện có $P_{\mathrm{MLE}}=0$. Nếu một token trong câu test có xác suất 0, xác suất toàn câu bằng 0 và perplexity vô hạn. Khi $n$ lớn, số lịch sử khả dĩ gia tăng nhanh, trong khi corpus luôn hữu hạn.

**Từ ngoài vốn từ (OOV):** xây vocab **chỉ trên train**, rồi ánh xạ token ngoài train vào `<unk>`. Trong ứng dụng thực tế, cần tạo một số `<unk>` ngay trong training (ví dụ dùng `min_count=2` trên corpus đủ lớn); nếu `<unk>` không có trong train thì ngay cả xác suất MLE cho `<unk>` vẫn bằng 0.

In [5]:
history = ('tôi', 'thích')
for w in ('học', 'ngôn', 'mô'):
    print(f'P_MLE({w} | {history}) = {mle_prob(w, history, 3):.3f}')
print('OOV demo:', prepare('tôi sáng_tạo câu mới', 3, vocab))

P_MLE(học | ('tôi', 'thích')) = 0.333
P_MLE(ngôn | ('tôi', 'thích')) = 0.333
P_MLE(mô | ('tôi', 'thích')) = 0.000
OOV demo: ['<s>', '<s>', 'tôi', '<unk>', '<unk>', '<unk>', '</s>']


## 4. Smoothing: Add-$k$ (Laplace khi $k=1$)
Với $V$ là tập token **có thể dự đoán**, $|V|$ là kích thước từ vựng (gồm `</s>` và `<unk>`, không gồm `<s>`):

$$P_{\mathrm{add}\text{-}k}(w\mid h)=\frac{C(h,w)+k}{C(h)+k|V|},\qquad k>0.$$

Khi $C(h)=0$, phân phối trở thành đều trên $V$. Add-one là baseline dễ hiểu nhưng thường làm phẳng quá mạnh đối với LM; thử $k=0.1$ và $k=1$ để so sánh.

In [6]:
def add_k_prob(word, history, order, k=0.1):
    assert k > 0
    if word not in vocab:
        word = '<unk>'
    counts, hc = models[order]
    h = tuple(history[-(order-1):]) if order > 1 else ()
    return (counts[h + (word,)] + k) / (hc[h] + k * len(predictable))

for k in (1.0, 0.1):
    print(f'k={k}: seen={add_k_prob("học", history, 3, k):.4f}; '
          f'unseen={add_k_prob("mô", history, 3, k):.4f}')
    assert abs(sum(add_k_prob(w, history, 3, k) for w in predictable) - 1) < 1e-10
print('PASS: add-k là phân phối chuẩn hóa.')

k=1.0: seen=0.0769; unseen=0.0385
k=0.1: seen=0.2075; unseen=0.0189
PASS: add-k là phân phối chuẩn hóa.


## 5. Back-off và Linear Interpolation
**Back-off (trực giác):** dùng trigram nếu đủ bằng chứng, nếu không chuyển xuống bigram, sau đó unigram. **Quan trọng:** việc chuyển cấp đơn giản dùng trực tiếp MLE **không đảm bảo tổng xác suất bằng 1**. Back-off chuẩn cần *discounting + phân bổ khối xác suất*, như Katz back-off. Trong notebook, ta minh họa back-off bằng **mô hình thứ bậc theo ngữ cảnh**: chọn mức cao nhất có đủ số quan sát, dùng Add-$k$ tại mức đó. Cách này bảo đảm chuẩn hóa theo từng ngữ cảnh nhưng chỉ là bản đơn giản hóa để học, không phải Katz back-off.

**Interpolation:** luôn phối trộn các cấp:
$$P_{\mathrm{interp}}(w\mid h_2)=\lambda_3P_3(w\mid h_2)+\lambda_2P_2(w\mid h_1)+\lambda_1P_1(w),$$
$$\lambda_i\geq 0,\qquad \lambda_1+\lambda_2+\lambda_3=1.$$

Trong demo, các $P_i$ đều dùng Add-$k$ để vẫn định nghĩa tại lịch sử chưa gặp. Có thể điều chỉnh $\lambda$ trên dev set; tuyệt đối không chọn theo test set.

In [7]:
def backoff_demo_prob(word, history, min_history_count=2, k=0.1):
    # Chọn bậc dựa trên history (không dựa trên word), do đó mỗi hàng vẫn cộng = 1.
    for order in (3, 2, 1):
        h = tuple(history[-(order-1):]) if order > 1 else ()
        if models[order][1][h] >= min_history_count or order == 1:
            return add_k_prob(word, h, order, k)

def interp_prob(word, history, lambdas=(0.2, 0.3, 0.5), k=0.1):
    # Lambdas theo thứ tự: unigram, bigram, trigram.
    assert len(lambdas) == 3 and all(x >= 0 for x in lambdas)
    assert abs(sum(lambdas) - 1) < 1e-9
    return sum(lam * add_k_prob(word, history, order, k)
               for order, lam in enumerate(lambdas, start=1))

for method in (backoff_demo_prob, interp_prob):
    for history in [('tôi','thích'), ('lịch_sử','hoàn_toàn_mới')]:
        total = sum(method(w, history) for w in predictable)
        assert abs(total - 1) < 1e-10, (method.__name__, history, total)
        print(method.__name__, history, 'sum =', round(total, 8))

backoff_demo_prob ('tôi', 'thích') sum = 1.0
backoff_demo_prob ('lịch_sử', 'hoàn_toàn_mới') sum = 1.0
interp_prob ('tôi', 'thích') sum = 1.0
interp_prob ('lịch_sử', 'hoàn_toàn_mới') sum = 1.0


### Bài tập nhanh 1 — suy luận về dữ liệu thưa
1. Vì sao tăng từ bigram lên trigram có thể giúp nắm bắt ngữ cảnh tốt hơn nhưng làm số n-gram có count 0 tăng?
2. Vì sao dùng MLE để sinh văn bản có thể bị mắc kẹt tại một lịch sử chưa quan sát?
3. Phân biệt *back-off minh họa* ở đây và Katz back-off đúng nghĩa.

## 6. Sentence log-likelihood và Perplexity
Cho chuỗi đánh giá có $N$ **token mục tiêu được dự đoán** (kể cả `</s>`; không đếm `<s>`):

$$\log_2 P(W)=\sum_{t=1}^{N}\log_2 P(w_t\mid h_t).$$
$$\mathrm{PPL}(W)=2^{-\frac{1}{N}\sum_{t=1}^{N}\log_2 P(w_t\mid h_t)}.$$

Perplexity thấp thường tốt hơn khi **cùng tập test, cùng cách tokenize và cùng từ vựng**. Một token có $P=0$ dẫn đến $\mathrm{PPL}=\infty$. Không được để dữ liệu test rò rỉ vào train, vocab hoặc bước tối ưu hyperparameter.

In [8]:
def evaluate_ppl(sentences, prob_fn, order=3):
    log_sum, target_count = 0.0, 0
    for sentence in sentences:
        toks = prepare(sentence, order, vocab)
        for i in range(order-1, len(toks)):
            h = tuple(toks[i-order+1:i]) if order > 1 else ()
            w = toks[i]
            p = prob_fn(w, h)
            if p <= 0:
                return float('inf')
            log_sum += log2(p)
            target_count += 1
    return 2 ** (-log_sum / target_count)

methods = {
    'Unigram add-0.1': (lambda w, h: add_k_prob(w, h, 1, 0.1), 1),
    'Bigram add-0.1':  (lambda w, h: add_k_prob(w, h, 2, 0.1), 2),
    'Trigram MLE':     (lambda w, h: mle_prob(w, h, 3), 3),
    'Trigram add-1':   (lambda w, h: add_k_prob(w, h, 3, 1), 3),
    'Trigram add-0.1': (lambda w, h: add_k_prob(w, h, 3, 0.1), 3),
    'Back-off demo':   (backoff_demo_prob, 3),
    'Interpolation':   (interp_prob, 3),
}
print(f'{"Model":22} {"Dev PPL":>12} {"Test PPL":>12}')
for label, (fn, order) in methods.items():
    print(f'{label:22} {evaluate_ppl(dev_sentences,fn,order):12.3f} '
          f'{evaluate_ppl(test_sentences,fn,order):12.3f}')
print('Chú ý: kết quả chỉ có tính minh họa vì corpus rất nhỏ.')

Model                       Dev PPL     Test PPL
Unigram add-0.1              14.861       17.466
Bigram add-0.1                3.056        3.530
Trigram MLE                     inf          inf
Trigram add-1                10.789       12.067
Trigram add-0.1               4.364        5.594
Back-off demo                 3.147        4.186
Interpolation                 4.183        4.862
Chú ý: kết quả chỉ có tính minh họa vì corpus rất nhỏ.


### Bài tập nhanh 2 — tính tay perplexity
Giả sử một LM gán xác suất lần lượt $[0.5,\;0.25,\;0.5]$ cho 3 token mục tiêu. Hãy tính:

$$\mathrm{PPL}=\left(\frac{1}{0.5\cdot 0.25\cdot 0.5}\right)^{1/3}.$$

Đáp số tham khảo sau khi tự tính: $\mathrm{PPL}=16^{1/3}\approx 2.520$.

In [9]:
p = [0.5, 0.25, 0.5]
calculated = 2 ** (-sum(log2(x) for x in p) / len(p))
print('PPL example =', round(calculated, 4))
assert abs(calculated - 16**(1/3)) < 1e-10

PPL example = 2.5198


## 7. Sinh văn bản: sampling từ phân phối $P(w\mid h)$
Quy trình: khởi tạo $n-1$ token `<s>`; tại mỗi bước lấy mẫu một token $w\sim P(\cdot\mid h)$; dịch cửa sổ và dừng nếu nhận `</s>` hoặc vượt số token tối đa.

**Temperature** $\tau>0$ thay đổi độ sắc của phân phối:
$$Q_{\tau}(w\mid h)=\frac{P(w\mid h)^{1/\tau}}{\sum_{v\in V}P(v\mid h)^{1/\tau}}.$$

$\tau<1$: thiên về từ xác suất cao; $\tau>1$: đa dạng hơn. Temperature biến đổi **phân phối khi sinh**, không thay đổi công thức PPL của mô hình gốc.

In [10]:
def generate(prob_fn, order=3, temperature=1.0, max_words=20, seed=42):
    if temperature <= 0:
        raise ValueError('temperature must be > 0')
    rng = random.Random(seed)
    context = ['<s>'] * (order - 1)
    output = []
    # Không phát sinh <unk> để đầu ra dễ đọc (chỉ là lựa chọn ở decoder).
    words = [w for w in predictable if w != '<unk>']
    for _ in range(max_words):
        h = tuple(context[-(order-1):]) if order > 1 else ()
        weights = [max(prob_fn(w, h), 0) ** (1 / temperature) for w in words]
        if sum(weights) <= 0:
            break
        next_word = rng.choices(words, weights=weights, k=1)[0]
        if next_word == '</s>':
            break
        output.append(next_word)
        context.append(next_word)
    return ' '.join(output)

for temperature in (0.6, 1.0, 1.5):
    print(f'τ={temperature}')
    for seed in (1, 2, 3):
        print(' ', generate(interp_prob, temperature=temperature, seed=seed))

τ=0.6
  bạn thích ngôn ngữ tự nhiên
  tôi thích học máy xử lý ngôn ngữ tự nhiên
  học máy
τ=1.0
  bạn tôi thú học ngôn ngữ tự nhiên
  tự xử
  học mô học nhiên nhiên
τ=1.5
  bạn từ tôi học ngữ mô rất tôi dữ
  xử xử bạn bạn từ thú thú lý ngữ rất rất dự mô lý thích đoán xử ngôn ngôn liệu
  học ngôn mô nhiên rất bạn


## 8. Thực hành có hướng dẫn (Guided Lab)
**Mục tiêu:** xây bigram/trigram, so sánh các kỹ thuật smoothing và sinh văn bản.

**Phần A (10 phút).** Chọn một câu mới, ghi ra unigram/bigram/trigram và xác suất từ tiếp theo. Kiểm tra kết quả bằng `models`.

**Phần B (15 phút).** Dùng `dev_sentences` để thử $k\in\{0.01,0.1,0.5,1\}$ và chọn $k$ có dev perplexity thấp nhất. Chỉ sau khi chọn mới đánh giá trên test.

**Phần C (15 phút).** So sánh interpolation với $\lambda=(0.2,0.3,0.5)$ và $(0.1,0.2,0.7)$. Dùng 3 seed khác nhau để sinh văn bản. Phân tích chất lượng đầu ra.

In [11]:
# A — Hiển thị top continuation theo trigram.
def top_next_words(history, prob_fn=interp_prob, top_k=6):
    return sorted(((w, round(prob_fn(w, history), 4)) for w in predictable),
                  key=lambda item: item[1], reverse=True)[:top_k]
print('Top continuation sau "tôi thích":')
print(top_next_words(('tôi','thích')))

# B — Hyperparameter search chỉ trên dev set.
k_candidates = (0.01, 0.1, 0.5, 1.0)
score_by_k = {k: evaluate_ppl(dev_sentences, lambda w,h,k=k: add_k_prob(w,h,3,k),3)
              for k in k_candidates}
best_k = min(score_by_k, key=score_by_k.get)
print('Dev PPL:', score_by_k)
print('Best k (dev):', best_k)
print('One-time Test PPL:', evaluate_ppl(test_sentences,
      lambda w,h: add_k_prob(w,h,3,best_k), 3))

# C — So sánh lambda trên dev; không tinh chỉnh dựa trên test.
for lam in [(0.2,0.3,0.5), (0.1,0.2,0.7)]:
    fn = lambda w,h,lam=lam: interp_prob(w,h,lam)
    print('lambda=', lam, 'Dev PPL=', round(evaluate_ppl(dev_sentences,fn,3), 3),
          '| generated:', generate(fn, seed=7))

Top continuation sau "tôi thích":
[('học', 0.2211), ('ngôn', 0.1735), ('xử', 0.165), ('</s>', 0.0429), ('ngữ', 0.0316), ('thích', 0.0259)]
Dev PPL: {0.01: 3.562965313194618, 0.1: 4.364406545348736, 0.5: 7.996162580361689, 1.0: 10.78949636851793}
Best k (dev): 0.01
One-time Test PPL: 4.995885015545748
lambda= (0.2, 0.3, 0.5) Dev PPL= 4.183 | generated: lý dữ ngôn bạn ngữ liệu
lambda= (0.1, 0.2, 0.7) Dev PPL= 4.103 | generated: máy bạn thích dữ ngôn mô bạn ngữ


## 9. Bài tập sinh viên (nộp báo cáo / Colab)

### Bài 1 — N-gram + MLE (2 điểm)
- Dùng corpus có ít nhất 30 câu của nhóm (có thể thử từ văn bản *Truyện Kiều*).
- Tách dữ liệu ở **mức câu** thành train/dev/test, rồi mới xây vocabulary từ train.
- In 10 unigram, bigram, trigram phổ biến nhất; tính $P(w\mid h)$ bằng MLE cho 3 ví dụ tự chọn.

### Bài 2 — Smoothing (2 điểm)
- Tự cài `add_k_prob` với $k=0.1$ và $k=1$; giải thích tại sao cần cộng $k|V|$ ở mẫu số.
- Kiểm tra $\sum_{w\in V}P(w\mid h)=1$ cho 5 lịch sử, gồm ít nhất 1 lịch sử chưa xuất hiện.

### Bài 3 — Back-off & Interpolation (2 điểm)
- So sánh quy tắc back-off minh họa với linear interpolation; ghi rõ giới hạn của back-off đơn giản.
- Tinh chỉnh $\lambda$ theo dev perplexity và giải thích lựa chọn.

### Bài 4 — Evaluation (2 điểm)
- Báo cáo bảng perplexity của unigram/bigram/trigram với MLE, Add-$k$ và interpolation.
- Giữ nguyên test set, tokenization, vocab và quy tắc tính số token để so sánh công bằng.
- Trình bày trường hợp perplexity vô hạn của MLE.

### Bài 5 — Text Generation (2 điểm)
- Sinh ít nhất 5 câu với trigram và interpolation ở $\tau=0.7,1.0,1.5$.
- Nhận xét mức lặp, tính trôi chảy và sự khác nhau về tính đa dạng.
- **Mở rộng:** với *Truyện Kiều*, sinh 2 cặp câu 6–8 âm tiết; đây là ràng buộc độ dài, **không** tự động bảo đảm niêm luật/vần của thơ lục bát.

**Yêu cầu nộp:** file `.ipynb` chạy được, kết quả bảng/biểu đồ và báo cáo ngắn (PDF) giải thích thí nghiệm. Ghi random seed và mô tả nguồn dữ liệu.

In [12]:
# ==============================================================================
# TODO 1: Thay bằng corpus của nhóm, chia train/dev/test ở mức câu và tạo vocab
# ==============================================================================
# Corpus gồm 36 câu thơ lục bát từ tác phẩm Truyện Kiều (Nguyễn Du)
student_corpus = [
    'trăm năm trong cõi người ta',
    'chữ tài chữ mệnh khéo là ghét nhau',
    'trải qua một cuộc bể dâu',
    'những điều trông thấy mà đau đớn lòng',
    'lạ gì bỉ sắc tư phong',
    'trời xanh quen thói má hồng đánh ghen',
    'cảo thơm lần giở trước đèn',
    'phong tình có lục còn truyền sử xanh',
    'rằng năm gia tĩnh triều minh',
    'bốn phương phẳng lặng hai kinh vững vàng',
    'có nhà viên ngoại họ vương',
    'gia tư nghĩ cũng thường thường bậc trung',
    'một trai con thứ rốt lòng',
    'vương quan là chữ nối dòng nho gia',
    'đầu lòng hai ả tố nga',
    'thúy kiều là chị em là thúy vân',
    'mai cốt cách tuyết tinh thần',
    'mỗi người một vẻ mười phân vẹn mười',
    'vân xem trang trọng khác vời',
    'khuôn trăng đầy đặn nét ngài nở nang',
    'hoa cười ngọc thốt đoan trang',
    'mây thua nước tóc tuyết nhường màu da',
    'kiều càng sắc sảo mặn mà',
    'so bề tài sắc lại là phần hơn',
    'làn thu thủy nét xuân sơn',
    'hoa ghen thua thắm liễu hờn kém xanh',
    'một hai nghiêng nước nghiêng thành',
    'sắc đành đòi một tài đành họa hai',
    'thông minh vốn sẵn tính trời',
    'pha nghề thi họa đủ mùi ca ngâm',
    'cung thương lầu bậc ngũ âm',
    'nghề riêng ăn đứt hồ cầm một trương',
    'khúc nhà tay lựa nên chương',
    'một thiên bạc mệnh lại càng não nhân',
    'phong lưu rất mực hồng quần',
    'xuân xanh xấp xỉ tới tuần cập kê',
]

# Chia dữ liệu ở mức câu thành train / dev / test
random.seed(SEED)
shuffled_corpus = list(student_corpus)
random.shuffle(shuffled_corpus)

train_student = shuffled_corpus[:28]   # 28 câu (~78%)
dev_student   = shuffled_corpus[28:32]  # 4 câu (~11%)
test_student  = shuffled_corpus[32:]    # 4 câu (~11%)

# Xây dựng từ vựng CHỈ TỪ tập train (tránh rò rỉ dữ liệu test)
vocab_student = make_vocabulary(train_student, min_count=1)
predictable_student = sorted(vocab_student)

print(f'=== KẾT QUẢ TODO 1: CHIA DỮ LIỆU & XÂY DỰNG VOCABULARY ===')
print(f'Tổng số câu corpus       : {len(student_corpus)}')
print(f'Số câu Train / Dev / Test : {len(train_student)} / {len(dev_student)} / {len(test_student)}')
print(f'Kích thước Vocabulary     : {len(vocab_student)} tokens (bao gồm </s> và <unk>)')
print(f'Ví dụ câu Train [0]       : "{train_student[0]}"')
print(f'Ví dụ câu Dev   [0]       : "{dev_student[0]}"')
print(f'Ví dụ câu Test  [0]       : "{test_student[0]}"')

# ==============================================================================
# TODO 2: Tìm 10 n-gram phổ biến nhất của corpus sinh viên (n=1, 2, 3)
# ==============================================================================
student_models = {order: fit_counts(train_student, order, vocab_student) for order in (1, 2, 3)}

print('\n=== KẾT QUẢ TODO 2: TOP 10 N-GRAM TRÊN TẬP HUẤN LUYỆN ===')
for order, name in [(1, 'UNIGRAM'), (2, 'BIGRAM'), (3, 'TRIGRAM')]:
    counts, _ = student_models[order]
    print(f'\n--- Top 10 {name} (Order {order}) ---')
    print(f'{"STT":<4} | {"N-gram":<30} | {"Số lần xuất hiện":<15}')
    print('-' * 55)
    for rank, (gram, cnt) in enumerate(counts.most_common(10), 1):
        gram_str = ' '.join(gram)
        print(f'{rank:<4} | {gram_str:<30} | {cnt:<15}')

# ==============================================================================
# TODO 3: Train ba mô hình (n=1,2,3) và tính xác suất MLE
# ==============================================================================
def student_mle_prob(word, history, order):
    counts, history_counts = student_models[order]
    h = tuple(history[-(order-1):]) if order > 1 else ()
    denom = history_counts[h]
    return counts[h + (word,)] / denom if denom else 0.0

print('\n=== KẾT QUẢ TODO 3: HUẤN LUYỆN MÔ HÌNH MLE (n=1, 2, 3) & VÍ DỤ XÁC SUẤT ===')
for order in (1, 2, 3):
    counts, hc = student_models[order]
    print(f'Mô hình Order={order}: {len(counts)} n-gram types, {sum(counts.values())} sự kiện dự đoán (prediction events)')

examples = [
    ('một', (), 1, 'P_MLE(một) [Unigram]'),
    ('một', ('<s>',), 2, 'P_MLE(một | <s>) [Bigram]'),
    ('phương', ('bốn',), 2, 'P_MLE(phương | bốn) [Bigram]'),
    ('phẳng', ('bốn', 'phương'), 3, 'P_MLE(phẳng | bốn, phương) [Trigram]'),
]

print('\n--- Các ví dụ tính xác suất P(w | h) bằng MLE ---')
for word, hist, ord_val, desc in examples:
    prob = student_mle_prob(word, hist, ord_val)
    counts, hc = student_models[ord_val]
    h_tuple = tuple(hist[-(ord_val-1):]) if ord_val > 1 else ()
    c_hw = counts[h_tuple + (word,)]
    c_h = hc[h_tuple]
    print(f'{desc:<38}: C({h_tuple + (word,)}) / C({h_tuple}) = {c_hw}/{c_h} = {prob:.4f}')

# Kiểm chứng chuẩn hóa phân phối xác suất trên 5 lịch sử đầu
for order in (1, 2, 3):
    _, hc = student_models[order]
    for history in list(hc.keys())[:5]:
        total = sum(student_mle_prob(w, history, order) for w in predictable_student)
        assert abs(total - 1.0) < 1e-9, f'Lỗi chuẩn hóa tại order={order}, history={history}'
print('Đã kiểm chứng: Tổng xác suất sum_{w in V} P_MLE(w | h) = 1.0 trên các lịch sử đã quan sát.')

# ==============================================================================
# TODO 4: So sánh Add-k, back-off minh họa, interpolation và test perplexity
# ==============================================================================
# 1. Cài đặt Add-k Smoothing (Bài 2)
def student_add_k_prob(word, history, order, k=0.1):
    if word not in vocab_student:
        word = '<unk>'
    counts, hc = student_models[order]
    h = tuple(history[-(order-1):]) if order > 1 else ()
    # Mẫu số cộng k * |V| để đảm bảo tổng xác suất bằng 1
    return (counts[h + (word,)] + k) / (hc[h] + k * len(predictable_student))

# Kiểm tra sum = 1 trên 5 lịch sử (gồm ít nhất 1 lịch sử chưa xuất hiện)
test_histories = [
    ('bốn', 'phương'),    # Đã xuất hiện trong train
    ('<s>', 'một'),       # Đã xuất hiện trong train
    ('nghiêng', 'nước'),  # Đã xuất hiện trong train
    ('thường', 'thường'), # Đã xuất hiện trong train
    ('lịch_sử', 'mới'),   # CHƯA từng xuất hiện trong train (unseen history)
]
print('\n=== KẾT QUẢ TODO 4: KIỂM CHỨNG ADD-k SMOOTHING & CHUẨN HÓA ===')
for hist in test_histories:
    for k_val in (0.1, 1.0):
        tot = sum(student_add_k_prob(w, hist, 3, k_val) for w in predictable_student)
        assert abs(tot - 1.0) < 1e-9
    print(f'Lịch sử {str(hist):<25}: sum(P_add_k) = 1.0 (k=0.1 & k=1.0) [PASS]')

# 2. Cài đặt Back-off minh họa & Linear Interpolation (Bài 3)
def student_backoff_prob(word, history, min_history_count=2, k=0.1):
    for order in (3, 2, 1):
        h = tuple(history[-(order-1):]) if order > 1 else ()
        if student_models[order][1][h] >= min_history_count or order == 1:
            return student_add_k_prob(word, h, order, k)

def student_interp_prob(word, history, lambdas=(0.1, 0.4, 0.5), k=0.1):
    assert len(lambdas) == 3 and abs(sum(lambdas) - 1.0) < 1e-9
    return sum(lam * student_add_k_prob(word, history, order, k)
               for order, lam in enumerate(lambdas, start=1))

# 3. Hàm tính Perplexity độc lập trên tập sinh viên
def student_evaluate_ppl(sentences, prob_fn, order=3):
    log_sum, target_count = 0.0, 0
    for sentence in sentences:
        toks = prepare(sentence, order, vocab_student)
        for i in range(order - 1, len(toks)):
            h = tuple(toks[i - order + 1:i]) if order > 1 else ()
            w = toks[i]
            p = prob_fn(w, h)
            if p <= 0:
                return float('inf')
            log_sum += log2(p)
            target_count += 1
    return 2 ** (-log_sum / target_count)

# Tinh chỉnh lambda trên dev set
best_dev_ppl = float('inf')
best_lambdas = None
for l1 in [0.1, 0.2, 0.3]:
    for l2 in [0.2, 0.3, 0.4]:
        l3 = round(1.0 - l1 - l2, 2)
        if l3 > 0:
            cand = (round(l1, 2), round(l2, 2), l3)
            ppl = student_evaluate_ppl(dev_student, lambda w, h, lam=cand: student_interp_prob(w, h, lam), 3)
            if ppl < best_dev_ppl:
                best_dev_ppl = ppl
                best_lambdas = cand

print(f'Bộ tham số tối ưu tìm được trên Dev: lambda = {best_lambdas} với Dev PPL = {best_dev_ppl:.3f}')

# 4. Bảng tổng hợp so sánh Perplexity (Bài 4)
methods_eval = {
    'Unigram MLE':         (lambda w, h: student_mle_prob(w, h, 1), 1),
    'Unigram Add-0.1':     (lambda w, h: student_add_k_prob(w, h, 1, 0.1), 1),
    'Unigram Add-1.0':     (lambda w, h: student_add_k_prob(w, h, 1, 1.0), 1),
    'Bigram MLE':          (lambda w, h: student_mle_prob(w, h, 2), 2),
    'Bigram Add-0.1':      (lambda w, h: student_add_k_prob(w, h, 2, 0.1), 2),
    'Bigram Add-1.0':      (lambda w, h: student_add_k_prob(w, h, 2, 1.0), 2),
    'Trigram MLE':         (lambda w, h: student_mle_prob(w, h, 3), 3),
    'Trigram Add-0.1':     (lambda w, h: student_add_k_prob(w, h, 3, 0.1), 3),
    'Trigram Add-1.0':     (lambda w, h: student_add_k_prob(w, h, 3, 1.0), 3),
    'Back-off demo':       (student_backoff_prob, 3),
    f'Interpolation {best_lambdas}': (lambda w, h: student_interp_prob(w, h, best_lambdas), 3),
}

print('\n=== BẢNG SO SÁNH PERPLEXITY (DEV & TEST) ===')
print(f'{"Mô hình":<32} | {"Dev PPL":>10} | {"Test PPL":>10}')
print('-' * 58)
for label, (fn, order) in methods_eval.items():
    dev_p = student_evaluate_ppl(dev_student, fn, order)
    test_p = student_evaluate_ppl(test_student, fn, order)
    dev_str = f'{dev_p:10.3f}' if dev_p < 1e6 else f'{"inf":>10}'
    test_str = f'{test_p:10.3f}' if test_p < 1e6 else f'{"inf":>10}'
    print(f'{label:<32} | {dev_str} | {test_str}')

print('\nGiải thích trường hợp MLE có PPL = inf: Do trong tập Dev/Test có N-gram chưa từng thấy trong Train,')
print('khiến xác suất P_MLE = 0 => log2(0) = -inf => PPL = 2^(-log_sum/N) = inf.')

# ==============================================================================
# TODO 5: Sinh câu ở tau = 0.7, 1.0, 1.5 & Sinh 2 cặp câu lục bát (Bài 5)
# ==============================================================================
def student_generate(prob_fn=None, order=3, temperature=1.0, max_words=20, seed=42, fixed_length=None):
    if temperature <= 0:
        raise ValueError('temperature must be > 0')
    if prob_fn is None:
        prob_fn = lambda w, h: student_interp_prob(w, h, best_lambdas)
    rng = random.Random(seed)
    context = ['<s>'] * (order - 1)
    output = []
    candidates = [w for w in predictable_student if w != '<unk>']
    target_len = fixed_length if fixed_length else max_words
    for _ in range(target_len):
        h = tuple(context[-(order - 1):]) if order > 1 else ()
        step_candidates = [w for w in candidates if w != '</s>'] if fixed_length else candidates
        weights = [max(prob_fn(w, h), 0) ** (1 / temperature) for w in step_candidates]
        if sum(weights) <= 0:
            break
        next_word = rng.choices(step_candidates, weights=weights, k=1)[0]
        if next_word == '</s>':
            break
        output.append(next_word)
        context.append(next_word)
    return ' '.join(output)

print('\n=== KẾT QUẢ TODO 5: SINH VĂN BẢN VỚI INTERPOLATION & ĐÁNH GIÁ NHIỆT ĐỘ TAU ===')
for tau in (0.7, 1.0, 1.5):
    print(f'\n--- Nhiệt độ tau = {tau} ---')
    for i in range(1, 6):
        gen_sent = student_generate(temperature=tau, seed=42 + i * 7)
        print(f'  Câu {i}: {gen_sent}')

print('\n--- MỞ RỘNG BÀI 5: SINH 2 CẶP CÂU LỤC BÁT (6 - 8 ÂM TIẾT) ---')
for pair_idx in (1, 2):
    cau_6 = student_generate(temperature=0.8, seed=200 + pair_idx * 3, fixed_length=6)
    cau_8 = student_generate(temperature=0.8, seed=200 + pair_idx * 3 + 1, fixed_length=8)
    print(f'Cặp lục bát {pair_idx}:')
    print(f'  (6 chữ): {cau_6}')
    print(f'  (8 chữ): {cau_8}')

print('\n--- NHẬN XÉT ĐÁNH GIÁ (BÀI 5) ---')
print('1. Tính lặp và trôi chảy (Fluency & Repetition):')
print('   - Tại tau = 0.7: Mô hình thiên về các từ có xác suất cao, sinh câu ngắn, trôi chảy cục bộ tốt nhưng ít biến hóa.')
print('   - Tại tau = 1.0: Cân bằng tốt giữa độ tự nhiên và vốn từ đa dạng.')
print('   - Tại tau = 1.5: Phân phối bị san phẳng, xuất hiện nhiều từ hiếm gặp, câu đa dạng nhất nhưng tính mạch lạc giảm.')
print('2. Thơ lục bát mở rộng:')
print('   - Ràng buộc độ dài (fixed_length=6, 8) đảm bảo số lượng âm tiết chuẩn thể thơ.')
print('   - Tuy nhiên mô hình n-gram thống kê chưa học được niêm luật bằng - trắc và gieo vần của thơ lục bát.')


=== KẾT QUẢ TODO 1: CHIA DỮ LIỆU & XÂY DỰNG VOCABULARY ===
Tổng số câu corpus       : 36
Số câu Train / Dev / Test : 28 / 4 / 4
Kích thước Vocabulary     : 161 tokens (bao gồm </s> và <unk>)
Ví dụ câu Train [0]       : "sắc đành đòi một tài đành họa hai"
Ví dụ câu Dev   [0]       : "đầu lòng hai ả tố nga"
Ví dụ câu Test  [0]       : "mai cốt cách tuyết tinh thần"

=== KẾT QUẢ TODO 2: TOP 10 N-GRAM TRÊN TẬP HUẤN LUYỆN ===

--- Top 10 UNIGRAM (Order 1) ---
STT  | N-gram                         | Số lần xuất hiện
-------------------------------------------------------
1    | </s>                           | 28             
2    | một                            | 6              
3    | hai                            | 3              
4    | xanh                           | 3              
5    | gia                            | 3              
6    | trời                           | 2              
7    | hồng                           | 2              
8    | ghen                         

## 10. Câu hỏi tự kiểm tra
1. Chain rule khác giả định Markov ở điểm nào?
2. Với $C(\text{"tôi học"})=3$ và $C(\text{"tôi học máy"})=2$, ước lượng $P(\text{"máy"}\mid\text{"tôi học"})$ bằng MLE là bao nhiêu?
3. Với $|V|=100$, $C(h)=10$, $C(h,w)=0$ và $k=1$, tính $P_{\mathrm{add}\text{-}1}(w\mid h)$.
4. Vì sao không được xây vocabulary từ cả train và test?
5. Tại sao perplexity nhỏ hơn không chứng minh chắc chắn chất lượng câu sinh tốt hơn?

<details><summary>Gợi ý đáp án (mở sau khi làm)</summary>

1. Chain rule chính xác; Markov là xấp xỉ chỉ sử dụng cửa sổ lịch sử hữu hạn.
2. $2/3$.
3. $1/(10+100)=1/110$.
4. Gây rò rỉ thông tin test và làm đánh giá sai lệch.
5. PPL đo likelihood trên dữ liệu, còn chất lượng sinh cần thêm tiêu chí trôi chảy, tính hợp lý, đa dạng và đánh giá thực tế.
</details>

## Tài liệu tham khảo
- Daniel Jurafsky & James H. Martin, *Speech and Language Processing*, 3rd ed. draft (2023), Ch. 3: **N-gram Language Models**.
- Phạm Tiến Lâm, *Language Models* (bài giảng học phần NLP), các phần n-gram, evaluation, OOV, smoothing, interpolation.
- Bài thực hành **Lab3**: N-gram và sinh thơ với *Truyện Kiều*; **Lab4**: add-one, back-off và interpolation.

**Ghi chú giảng dạy:** Triển khai từ mô hình đếm đơn giản đến đánh giá trên dữ liệu chưa thấy; phân biệt cụ thể *mô phỏng back-off dễ học* và *Katz back-off có discounting/normalization*.